# 第 2 周 · NumPy 矩阵与梯度实验

本文使用 Python 3.12 和 NumPy 2.2.6。从首个单元格顺序执行，断言全部通过即可完成实验。数据写在下方，安装依赖后可离线运行。

## 1. 定义数据

两条样本各有三个特征，权重矩阵的两列对应两项输出。

In [ ]:
import numpy as np

X = np.array([[1, 2, 0], [0, 1, 3]], dtype=np.float64)
W = np.array([[1, 0], [2, -1], [0, 1]], dtype=np.float64)
b = np.array([1, -1], dtype=np.float64)

## 2. 核对矩阵乘法

先断言形状，再拿乘积和加偏置的结果与手算对照。

In [ ]:
product = X @ W
scores = product + b

assert product.shape == scores.shape == (2, 2)
np.testing.assert_array_equal(product, [[5, -2], [2, 2]])
np.testing.assert_array_equal(scores, [[6, -3], [3, 1]])
print(scores)

## 3. 对比两种残差

切片保留的列轴会触发两两相减。用整数索引取得向量后，残差按样本一对一计算。

In [ ]:
y = np.array([4, 2], dtype=np.float64)
wrong = scores[:, :1] - y
residual = scores[:, 0] - y

assert wrong.shape == (2, 2)
assert residual.shape == y.shape == (2,)
np.testing.assert_array_equal(wrong, [[2, 4], [-1, 1]])
np.testing.assert_array_equal(residual, [2, 1])
print(wrong)

## 4. 检查梯度

用半平方和定义损失，用中心差分逐个核对三个权重的偏导。

In [ ]:
w = W[:, 0]


def loss(weights):
    r = X @ weights + b[0] - y
    return 0.5 * (r @ r)


def central_difference(weights, step):
    gradient = np.zeros_like(weights)
    for i in range(weights.size):
        offset = np.zeros_like(weights)
        offset[i] = step
        upper = loss(weights + offset)
        lower = loss(weights - offset)
        gradient[i] = (upper - lower) / (2 * step)
    return gradient


analytic = X.T @ residual
numeric = central_difference(w, step=1e-5)

assert loss(w) == 2.5
np.testing.assert_array_equal(analytic, [2, 5, 3])
np.testing.assert_allclose(numeric, analytic, rtol=0, atol=1e-8)
print(analytic)
print(numeric)

## 5. 检查太小的步长

观察第二个权重向两侧扰动后是否仍等于原值。

In [ ]:
step = 1e-16
assert w[1] + step == w[1] - step == w[1]
too_small = central_difference(w, step)
assert too_small[1] == 0.0
print(f"第二个权重的差分估计 {too_small[1]:.1f}")

## 6. 观察指数溢出

比较直接求指数、减去最大值和 `np.logaddexp`。

In [ ]:
pair = np.array([1000, 1001], dtype=np.float64)

with np.errstate(over="ignore"):
    naive = np.log(np.exp(pair).sum())

largest = pair.max()
shifted = largest + np.log(np.exp(pair - largest).sum())
stable = np.logaddexp(pair[0], pair[1])

assert np.isposinf(naive)
assert np.isfinite(stable)
np.testing.assert_allclose(shifted, stable, rtol=0, atol=1e-12)
print(f"直接计算 {naive}")
print(f"稳定计算 {stable:.8f}")